# 02 · Ejercicio 1: tipo de cambio del Banguat
Método del WSDL: `TipoCambioRango(fechainit, fechafin)`, con fechas como texto `dd/mm/yyyy`.
Responde `Vars/Var` con `moneda`, `fecha`, `venta` y `compra`. El cliente SOAP está en `src/banguat.py`.

In [0]:
%run ./_config

In [0]:
import uuid
from datetime import date, datetime, timezone

from pyspark.sql import functions as F

from src import banguat

## Rango de fechas de las transacciones

In [0]:
rango = spark.sql(f"""
    SELECT to_date(min(ts)) AS fecha_ini,
           to_date(max(ts)) AS fecha_fin,
           count_if(ts IS NULL) AS no_parseables
    FROM (SELECT try_to_timestamp(trim(fecha_hora), 'yyyy-MM-dd HH:mm:ss') AS ts
          FROM {BRONZE}.transacciones_raw)
""").first()

FECHA_INI, FECHA_FIN = rango.fecha_ini, rango.fecha_fin
print(f"Rango de transacciones: {FECHA_INI} a {FECHA_FIN}")
if rango.no_parseables:
    print(f"{rango.no_parseables} transacciones con fecha_hora no interpretable quedan fuera del rango")

Rango de transacciones: 2025-06-01 a 2025-06-10


In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}.tipo_cambio_raw (
        id_consulta    STRING,
        fecha_consulta TIMESTAMP COMMENT 'UTC',
        fecha_ini      DATE,
        fecha_fin      DATE,
        endpoint       STRING,
        http_status    INT,
        intentos       INT,
        error          STRING,
        xml_crudo      STRING
    )
    COMMENT 'Respuestas del servicio TipoCambioRango del Banguat, una fila por consulta'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}.tipo_cambio (
        fecha          DATE          NOT NULL,
        tasa           DECIMAL(10,5) NOT NULL COMMENT 'Quetzales por 1 USD',
        id_consulta    STRING,
        fecha_consulta TIMESTAMP
    )
    COMMENT 'Tipo de cambio de referencia USD/GTQ del Banguat por día'
""")

DataFrame[]

In [0]:
ESQUEMA_BRONZE = spark.table(f"{BRONZE}.tipo_cambio_raw").schema


def registrar_bronze(fecha_ini, fecha_fin, *, respuesta=None, error=None):
    """Agrega una consulta a bronze.tipo_cambio_raw, exitosa o fallida."""
    id_consulta = str(uuid.uuid4())
    fila = (
        id_consulta,
        respuesta.fecha_consulta if respuesta else datetime.now(timezone.utc),
        fecha_ini,
        fecha_fin,
        respuesta.endpoint if respuesta else banguat.URL,
        respuesta.http_status if respuesta else None,
        respuesta.intentos if respuesta else None,
        error,
        respuesta.xml_crudo if respuesta else None,
    )
    spark.createDataFrame([fila], ESQUEMA_BRONZE).write.mode("append").saveAsTable(f"{BRONZE}.tipo_cambio_raw")
    return id_consulta

## Bronze: respuesta XML cruda

In [0]:
try:
    respuesta = banguat.consultar_tipo_cambio_rango(FECHA_INI, FECHA_FIN)
except banguat.BanguatError as e:
    registrar_bronze(FECHA_INI, FECHA_FIN, error=f"{type(e).__name__}: {e}")
    raise

id_consulta = registrar_bronze(FECHA_INI, FECHA_FIN, respuesta=respuesta)
print(f"Consulta {id_consulta}: HTTP {respuesta.http_status}, {respuesta.intentos} intento(s)")

Consulta 5ae87c23-164e-4b12-b77b-1fa1d68f6245: HTTP 200, 1 intento(s)


In [0]:
display(spark.table(f"{BRONZE}.tipo_cambio_raw").orderBy(F.desc("fecha_consulta")))

id_consulta,fecha_consulta,fecha_ini,fecha_fin,endpoint,http_status,intentos,error,xml_crudo
5ae87c23-164e-4b12-b77b-1fa1d68f6245,2026-10-05T20:29:21.592Z,2025-06-01,2025-06-10,https://www.banguat.gob.gt/variables/ws/TipoCambio.asmx,200,1,null,"<?xml version=""1.0"" encoding=""utf-8""?>201/06/20257.676447.67644202/06/20257.675687.67568203/06/20257.678677.67867204/06/20257.678837.67883205/06/20257.683227.68322206/06/20257.682437.68243207/06/20257.682437.68243208/06/20257.682437.68243209/06/20257.682887.68288210/06/20257.682867.6828610"


## Silver: tabla `tipo_cambio`

In [0]:
df_tc = banguat.parsear_tipo_cambio(respuesta.xml_crudo)
faltantes = banguat.validar_rango(df_tc, respuesta.fecha_ini, respuesta.fecha_fin)
if faltantes:
    print(f"Fechas sin tasa: {faltantes}")

implausibles = df_tc[(df_tc["tasa"] < 5) | (df_tc["tasa"] > 12)]
if not implausibles.empty:
    raise ValueError(f"Tasas fuera de rango:\n{implausibles}")

filas = [(r.fecha, r.tasa, id_consulta, respuesta.fecha_consulta) for r in df_tc.itertuples(index=False)]
(
    spark.createDataFrame(filas, "fecha DATE, tasa DECIMAL(10,5), id_consulta STRING, fecha_consulta TIMESTAMP")
    .createOrReplaceTempView("tipo_cambio_nuevo")
)

display(spark.sql(f"""
    MERGE INTO {SILVER}.tipo_cambio AS t
    USING tipo_cambio_nuevo AS s
    ON t.fecha = s.fecha
    WHEN MATCHED AND t.tasa <> s.tasa THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
10,0,0,10


In [0]:
display(spark.table(f"{SILVER}.tipo_cambio").orderBy("fecha"))

fecha,tasa,id_consulta,fecha_consulta
2025-06-01,7.67644,5ae87c23-164e-4b12-b77b-1fa1d68f6245,2026-10-05T20:29:21.592Z
2025-06-02,7.67568,5ae87c23-164e-4b12-b77b-1fa1d68f6245,2026-10-05T20:29:21.592Z
2025-06-03,7.67867,5ae87c23-164e-4b12-b77b-1fa1d68f6245,2026-10-05T20:29:21.592Z
2025-06-04,7.67883,5ae87c23-164e-4b12-b77b-1fa1d68f6245,2026-10-05T20:29:21.592Z
2025-06-05,7.68322,5ae87c23-164e-4b12-b77b-1fa1d68f6245,2026-10-05T20:29:21.592Z
2025-06-06,7.68243,5ae87c23-164e-4b12-b77b-1fa1d68f6245,2026-10-05T20:29:21.592Z
2025-06-07,7.68243,5ae87c23-164e-4b12-b77b-1fa1d68f6245,2026-10-05T20:29:21.592Z
2025-06-08,7.68243,5ae87c23-164e-4b12-b77b-1fa1d68f6245,2026-10-05T20:29:21.592Z
2025-06-09,7.68288,5ae87c23-164e-4b12-b77b-1fa1d68f6245,2026-10-05T20:29:21.592Z
2025-06-10,7.68286,5ae87c23-164e-4b12-b77b-1fa1d68f6245,2026-10-05T20:29:21.592Z


## Exportación del XML crudo y de `tipo_cambio`

In [0]:
with open(f"{EXPORTS_PATH}/tipo_cambio_raw.xml", "w", encoding="utf-8") as f:
    f.write(respuesta.xml_crudo)

(
    spark.table(f"{SILVER}.tipo_cambio").select("fecha", "tasa").orderBy("fecha")
    .toPandas().to_csv(f"{EXPORTS_PATH}/tipo_cambio.csv", index=False)
)
display(dbutils.fs.ls(EXPORTS_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/gold/exports/dq_reporte.csv,dq_reporte.csv,1968,1791229133000
dbfs:/Volumes/workspace/gold/exports/tipo_cambio.csv,tipo_cambio.csv,201,1791232178000
dbfs:/Volumes/workspace/gold/exports/tipo_cambio_raw.xml,tipo_cambio_raw.xml,1432,1791232178000
dbfs:/Volumes/workspace/gold/exports/transacciones_gtq.csv,transacciones_gtq.csv,965,1791229133000


## Manejo de errores

In [0]:
casos = {
    "timeout (0.001 s, 2 intentos)": lambda: banguat.consultar_tipo_cambio_rango(
        FECHA_INI, FECHA_FIN, timeout=0.001, max_intentos=2, backoff_seg=0.5),
    "endpoint inexistente (HTTP 500, 2 intentos)": lambda: banguat.consultar_tipo_cambio_rango(
        FECHA_INI, FECHA_FIN, url=banguat.URL.replace("TipoCambio.asmx", "NoExiste.asmx"),
        max_intentos=2, backoff_seg=0.5),
    "XML inválido": lambda: banguat.parsear_tipo_cambio("<html>Service Unavailable</html"),
    "sin datos (rango futuro)": lambda: banguat.obtener_tipo_cambio(date(2030, 6, 1), date(2030, 6, 10)),
}
for nombre, caso in casos.items():
    try:
        caso()
        print(f"{nombre}: sin error")
    except banguat.BanguatError as e:
        print(f"{nombre}: {type(e).__name__} → {str(e)[:150]}")

Intento 1/2 falló: HTTPSConnectionPool(host='www.banguat.gob.gt', port=443): Max retries exceeded with url: /variables/ws/TipoCambio.asmx (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0xff314f146c60>, 'Connection to www.banguat.gob.gt timed out. (connect timeout=0.001)'))
Intento 2/2 falló: HTTPSConnectionPool(host='www.banguat.gob.gt', port=443): Max retries exceeded with url: /variables/ws/TipoCambio.asmx (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0xff314f097ad0>, 'Connection to www.banguat.gob.gt timed out. (connect timeout=0.001)'))


timeout (0.001 s, 2 intentos): ServicioNoDisponibleError → Banguat no respondió correctamente después de 2 intentos: HTTPSConnectionPool(host='www.banguat.gob.gt', port=443): Max retries exceeded with url: /va


Intento 1/2 falló: HTTP 500: <!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta http-equiv="Content-Type" content="text/html; charset=iso-8859-1"/>
<title>500 - Internal server error.</title>
<style type="tex
Intento 2/2 falló: HTTP 500: <!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta http-equiv="Content-Type" content="text/html; charset=iso-8859-1"/>
<title>500 - Internal server error.</title>
<style type="tex


endpoint inexistente (HTTP 500, 2 intentos): ServicioNoDisponibleError → Banguat no respondió correctamente después de 2 intentos: HTTP 500: <!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xht
XML inválido: XMLInvalidoError → La respuesta no es XML válido: unclosed token: line 1, column 25
sin datos (rango futuro): SinDatosError → La respuesta no trae tasas para el rango consultado
